In [1]:
# ── CELL 1: Install ───────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q",
    "openai", "pinecone", "groq",
    "sentence-transformers", "rank-bm25", "scikit-learn"
], check=True)
print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 12.3 MB/s eta 0:00:00
✓ Dependencies installed


In [2]:
# ── CELL 2: Imports ───────────────────────────────────────────
import os, json, pickle, time, warnings
import numpy as np
from openai import OpenAI
from pinecone import Pinecone
from groq import Groq
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from kaggle_secrets import UserSecretsClient
warnings.filterwarnings("ignore")
print("✓ Imports done")

✓ Imports done


In [3]:
# ── CELL 3: Load secrets ──────────────────────────────────────
secrets      = UserSecretsClient()
OPENAI_KEY   = secrets.get_secret("OPENAI_API_KEY")
PINECONE_KEY = secrets.get_secret("PINECONE_API_KEY")
GROQ_KEY     = secrets.get_secret("GROQ_API_KEY")
print("✓ All secrets loaded")

✓ All secrets loaded


In [4]:
# ── CELL 4: Initialize clients ────────────────────────────────
openai_client = OpenAI(api_key=OPENAI_KEY)
pc            = Pinecone(api_key=PINECONE_KEY)
groq_client   = Groq(api_key=GROQ_KEY)
reranker      = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("✓ Clients initialized")
print("  Generation model : gpt-4o-mini (OpenAI)")
print("  Judge model      : llama-3.3-70b-versatile (Groq)")
print("  Reranker         : cross-encoder/ms-marco-MiniLM-L-6-v2")
print("  Embedding model  : text-embedding-3-small (OpenAI)")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✓ Clients initialized
  Generation model : gpt-4o-mini (OpenAI)
  Judge model      : llama-3.3-70b-versatile (Groq)
  Reranker         : cross-encoder/ms-marco-MiniLM-L-6-v2
  Embedding model  : text-embedding-3-small (OpenAI)


In [5]:
# ── CELL 5: Load all 3 BM25 corpora ──────────────────────────
BASE = "/kaggle/input/datasets/rockybhai159/hello1233"
BM25_INDEXES = {}
CHUNK_STORES = {}

for label, pkl_name in [
    ("A", "corpus_A_fixed.pkl"),
    ("B", "corpus_B_recursive.pkl"),
    ("C", "corpus_C_semantic.pkl"),
]:
    with open(f"{BASE}/{pkl_name}", "rb") as f:
        data = pickle.load(f)
    chunks   = data["chunks"]
    metadata = data["metadata"]
    BM25_INDEXES[label] = BM25Okapi([c.lower().split() for c in chunks])
    CHUNK_STORES[label] = {"chunks": chunks, "metadata": metadata}
    print(f"  ✓ BM25 Strategy {label}: {len(chunks)} chunks")

  ✓ BM25 Strategy A: 2007 chunks
  ✓ BM25 Strategy B: 2753 chunks
  ✓ BM25 Strategy C: 1455 chunks


In [6]:
# ── CELL 6: Connect Pinecone indexes ──────────────────────────
INDEXES = {
    "A": pc.Index("rag-index-a-fixed"),
    "B": pc.Index("rag-index-b-recursive"),
    "C": pc.Index("rag-index-c-semantic"),
}
print("✓ Pinecone indexes connected")
for key, idx in INDEXES.items():
    stats = idx.describe_index_stats()
    print(f"   Index {key}: {stats['total_vector_count']} vectors")

✓ Pinecone indexes connected
   Index A: 2007 vectors
   Index B: 2753 vectors
   Index C: 1455 vectors


In [7]:
# ── CELL 7: Retrieval functions ───────────────────────────────

def embed_query(query):
    """Embed query using text-embedding-3-small"""
    r = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query.replace("\n", " ")
    )
    return r.data[0].embedding


def retrieve(query, strategy="B", mode="hybrid", top_k=15, final_k=5):
    """
    strategy : A, B, C  — which Pinecone index + BM25 corpus
    mode     : semantic  → Pinecone only
               hybrid   → BM25 + Pinecone + RRF + Cross-Encoder
    """
    index = INDEXES[strategy]

    # Step 1: Semantic search
    t0    = time.time()
    q_vec = embed_query(query)
    s_res = index.query(vector=q_vec, top_k=top_k, include_metadata=True)
    ret_t = round(time.time() - t0, 3)

    p_chunks = [
        (r["metadata"]["text"], r["metadata"])
        for r in s_res["matches"]
        if "text" in r["metadata"]
    ]

    if mode == "semantic":
        return p_chunks[:final_k], ret_t

    # Step 2: BM25 — uses matching corpus for this strategy
    bm25_idx  = BM25_INDEXES[strategy]
    c_data    = CHUNK_STORES[strategy]
    b_scores  = bm25_idx.get_scores(query.lower().split())
    b_ids     = np.argsort(b_scores)[::-1][:top_k].tolist()
    b_chunks  = [
        (c_data["chunks"][i], c_data["metadata"][i])
        for i in b_ids if i < len(c_data["chunks"])
    ]

    # Step 3: RRF — Reciprocal Rank Fusion
    k = 60
    rrf_scores, chunk_store = {}, {}
    for rank, (chunk, meta) in enumerate(p_chunks):
        key = chunk[:80]
        rrf_scores[key]  = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_store[key] = (chunk, meta)
    for rank, (chunk, meta) in enumerate(b_chunks):
        key = chunk[:80]
        rrf_scores[key]  = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_store[key] = (chunk, meta)

    sorted_keys    = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    rrf_candidates = [chunk_store[k] for k in sorted_keys[:top_k]]

    # Step 4: Cross-Encoder reranking
    pairs  = [(query, c) for c, _ in rrf_candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, rrf_candidates), reverse=True)

    return [item for _, item in ranked[:final_k]], ret_t


def generate_answer(query, chunks):
    """Generate answer using gpt-4o-mini"""
    context = "\n\n---\n\n".join([
        f"[Source: {m.get('title','?')} ({m.get('year','?')})]\n{c}"
        for c, m in chunks
    ])
    prompt = f"""You are a research assistant specializing in machine learning.
Answer using ONLY the provided context from academic papers.
Do not fabricate facts or citations.
If context is insufficient, say so explicitly.

Context:
{context}

Question: {query}
Answer:"""

    t0  = time.time()
    res = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=512,
    )
    return res.choices[0].message.content, context, round(time.time()-t0, 3)


print("All functions defined")

All functions defined


In [8]:
# ── CELL 8: Evaluation functions ─────────────────────────────

def llm_judge(prompt, max_tokens=400):
    r = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content.strip()


def evaluate_faithfulness(answer, context):
    """
    Step 1: Extract all factual claims from the answer
    Step 2: Verify each claim against retrieved context
    Score  = verified claims / total claims
    """
    # Extract claims
    claims_raw = llm_judge(
        f"""Extract all distinct factual claims from this answer.
Return ONLY a numbered list, one claim per line. No preamble or explanation.

Answer: {answer}""", max_tokens=400)

    claims = []
    for line in claims_raw.strip().split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            claim = line.split(". ", 1)[-1].strip("- ").strip()
            if len(claim) > 15:
                claims.append(claim)

    if not claims:
        return 0.0, []

    # Verify each claim
    verified, results = 0, []
    for claim in claims:
        verdict = llm_judge(
            f"""Context from academic papers:
{context[:2500]}

Is this claim directly supported by the context above?
Answer with YES or NO only. No explanation.

Claim: {claim}""", max_tokens=5)

        supported = verdict.upper().startswith("YES")
        verified += int(supported)
        results.append({"claim": claim, "supported": supported})
        time.sleep(0.3)   # avoid rate limits

    return round(verified / len(claims), 3), results


def evaluate_relevancy(query, answer):
    """
    Step 1: Generate 3 questions from the answer
    Step 2: Compute cosine similarity with original query
    Score  = mean of 3 similarity scores
    """
    gen_qs_raw = llm_judge(
        f"""Generate exactly 3 questions that this answer is responding to.
Return ONLY the 3 questions, one per line. No numbering, no explanation.

Answer: {answer}""", max_tokens=200)

    gen_qs = [q.strip() for q in gen_qs_raw.strip().split("\n")
              if q.strip() and len(q.strip()) > 10][:3]

    if not gen_qs:
        return 0.0, []

    # Embed original query
    orig_emb = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    # Embed generated questions
    gen_embs = []
    for q in gen_qs:
        emb = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=q
        ).data[0].embedding
        gen_embs.append(emb)

    sims      = cosine_similarity([orig_emb], gen_embs)[0]
    avg_score = round(float(np.mean(sims)), 3)

    return avg_score, list(zip(gen_qs, [round(float(s), 3) for s in sims]))



print("All functions defined")

All functions defined


In [9]:
# ── CELL 9: Load golden set ───────────────────────────────────
GOLDEN_PATH = "/kaggle/input/datasets/rockybhai159/golden-set/golden_set.json"

with open(GOLDEN_PATH) as f:
    golden = json.load(f)

print(f"✓ Loaded {len(golden)} golden questions")
print(f"\n  Sample questions:")
for i, qa in enumerate(golden[:3]):
    print(f"  Q{i+1}: {qa['question'][:80]}")

✓ Loaded 30 golden questions

  Sample questions:
  Q1: How does the concept of Nested Learning relate to the idea of in-context learnin
  Q2: What is the significance of representing a machine learning model as a set of ne
  Q3: How do Fast Weight Programmers (FWPs) differ from conventional Recurrent Neural 


In [11]:
# ── CELL 10: Define 6 ablation configurations ─────────────────
CONFIGS = [
    {"id":1, "strategy":"A", "mode":"semantic",
     "label":"Fixed Chunking + Semantic Only"},
    {"id":2, "strategy":"B", "mode":"semantic",
     "label":"Recursive Chunking + Semantic Only"},
    {"id":3, "strategy":"C", "mode":"semantic",
     "label":"Semantic Chunking + Semantic Only"},
    {"id":4, "strategy":"A", "mode":"hybrid",
     "label":"Fixed Chunking + Hybrid + Rerank"},
    {"id":5, "strategy":"B", "mode":"hybrid",
     "label":"Recursive Chunking + Hybrid + Rerank"},
    {"id":6, "strategy":"C", "mode":"hybrid",
     "label":"Semantic Chunking + Hybrid + Rerank"},
]

print(f"\n{'='*60}")
print(f"  ABLATION STUDY — 6 configurations × {len(golden)} questions")
print(f"  Generation : gpt-4o-mini (OpenAI)")
print(f"  Judge      : gpt-4o-mini (OpenAI)")
print(f"{'='*60}")



  ABLATION STUDY — 6 configurations × 30 questions
  Generation : gpt-4o-mini (OpenAI)
  Judge      : gpt-4o-mini (OpenAI)


In [12]:
 # ── CELL 11: Run ablation study ───────────────────────────────
all_results   = {}
example_claims = {}   # store for report — 3 examples per config

for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Config {config['id']}: {config['label']}")
    print(f"{'='*60}")

    faith_scores = []
    rel_scores   = []
    ret_times    = []
    gen_times    = []
    config_examples = []

    for i, item in enumerate(golden):
        q = item["question"]

        try:
            # Retrieve
            chunks, ret_t = retrieve(
                q,
                strategy=config["strategy"],
                mode=config["mode"]
            )
            ret_times.append(ret_t)

            # Generate answer using gpt-4o-mini
            answer, context, gen_t = generate_answer(q, chunks)
            gen_times.append(gen_t)

            # Evaluate faithfulness using llama-3.3-70b
            faith, claims = evaluate_faithfulness(answer, context)
            faith_scores.append(faith)

            # Evaluate relevancy using text-embedding-3-small
            rel, gen_qs = evaluate_relevancy(q, answer)
            rel_scores.append(rel)

            # Save first 3 examples for report
            if i < 3:
                config_examples.append({
                    "question": q,
                    "answer":   answer,
                    "claims":   claims,
                    "faith":    faith,
                    "rel":      rel,
                    "sources":  [m.get("title","?") for _, m in chunks],
                })

            print(f"  Q{i+1:02d} | Faith={faith:.3f} | Rel={rel:.3f} | "
                  f"Ret={ret_t}s | Gen={gen_t}s")

            time.sleep(0.5)

        except Exception as e:
            print(f"  Q{i+1:02d} ERROR: {e}")
            faith_scores.append(0.0)
            rel_scores.append(0.0)
            if ret_times: ret_times.append(ret_times[-1])
            if gen_times: gen_times.append(gen_times[-1])
            time.sleep(3)

    # Store config results
    all_results[config["id"]] = {
        "label":             config["label"],
        "strategy":          config["strategy"],
        "mode":              config["mode"],
        "avg_faithfulness":  round(sum(faith_scores)/len(faith_scores), 3),
        "avg_relevancy":     round(sum(rel_scores)/len(rel_scores), 3),
        "avg_ret_ms":        round(sum(ret_times)/len(ret_times)*1000) if ret_times else 0,
        "avg_gen_ms":        round(sum(gen_times)/len(gen_times)*1000) if gen_times else 0,
        "avg_total_ms":      round((sum(ret_times)+sum(gen_times))/max(len(ret_times),1)*1000),
        "all_faith":         faith_scores,
        "all_rel":           rel_scores,
    }
    example_claims[config["id"]] = config_examples

    print(f"\n  Config {config['id']} Summary:")
    print(f"    Avg Faithfulness : {all_results[config['id']]['avg_faithfulness']}")
    print(f"    Avg Relevancy    : {all_results[config['id']]['avg_relevancy']}")
    print(f"    Avg Ret Time     : {all_results[config['id']]['avg_ret_ms']}ms")
    print(f"    Avg Gen Time     : {all_results[config['id']]['avg_gen_ms']}ms")


          


  Config 1: Fixed Chunking + Semantic Only
  Q01 | Faith=1.000 | Rel=0.826 | Ret=1.91s | Gen=3.752s
  Q02 | Faith=1.000 | Rel=0.663 | Ret=0.229s | Gen=3.403s
  Q03 | Faith=1.000 | Rel=0.758 | Ret=0.729s | Gen=2.644s
  Q04 | Faith=1.000 | Rel=0.737 | Ret=0.778s | Gen=1.894s
  Q05 | Faith=1.000 | Rel=0.754 | Ret=0.911s | Gen=2.932s
  Q06 | Faith=1.000 | Rel=0.699 | Ret=0.212s | Gen=3.439s
  Q07 | Faith=1.000 | Rel=0.676 | Ret=0.541s | Gen=1.847s
  Q08 | Faith=0.889 | Rel=0.550 | Ret=0.253s | Gen=2.452s
  Q09 | Faith=1.000 | Rel=0.723 | Ret=0.216s | Gen=2.569s
  Q10 | Faith=0.778 | Rel=0.549 | Ret=0.191s | Gen=2.575s
  Q11 | Faith=1.000 | Rel=0.630 | Ret=0.247s | Gen=1.525s
  Q12 | Faith=0.909 | Rel=0.675 | Ret=0.337s | Gen=3.361s
  Q13 | Faith=1.000 | Rel=0.711 | Ret=0.227s | Gen=4.274s
  Q14 | Faith=0.923 | Rel=0.697 | Ret=0.242s | Gen=2.746s
  Q15 | Faith=1.000 | Rel=0.591 | Ret=0.218s | Gen=2.417s
  Q16 | Faith=0.714 | Rel=0.673 | Ret=0.214s | Gen=2.986s
  Q17 | Faith=1.000 | Rel=0.7

In [13]:
# ── CELL 12: Print final results table ────────────────────────
print(f"\n{'='*80}")
print(f"  FINAL ABLATION RESULTS")
print(f"{'='*80}")
print(f"  {'#':<3} {'Configuration':<40} {'Faith':>7} {'Rel':>7} "
      f"{'Ret(ms)':>9} {'Gen(ms)':>9} {'Tot(ms)':>9}")
print(f"  {'-'*80}")

# Find best config
best_id = max(all_results,
              key=lambda x: all_results[x]["avg_faithfulness"] +
                            all_results[x]["avg_relevancy"])

for cid, r in all_results.items():
    marker = "  ← BEST" if cid == best_id else ""
    print(f"  {cid:<3} {r['label']:<40} "
          f"{r['avg_faithfulness']:>7} "
          f"{r['avg_relevancy']:>7} "
          f"{r['avg_ret_ms']:>9} "
          f"{r['avg_gen_ms']:>9} "
          f"{r['avg_total_ms']:>9}{marker}")

print(f"\n  Best configuration : Config {best_id} — {all_results[best_id]['label']}")
print(f"  Faithfulness       : {all_results[best_id]['avg_faithfulness']}")
print(f"  Relevancy          : {all_results[best_id]['avg_relevancy']}")


  FINAL ABLATION RESULTS
  #   Configuration                              Faith     Rel   Ret(ms)   Gen(ms)   Tot(ms)
  --------------------------------------------------------------------------------
  1   Fixed Chunking + Semantic Only             0.925   0.688       352      2721      3073
  2   Recursive Chunking + Semantic Only         0.964   0.696       270      4020      4290
  3   Semantic Chunking + Semantic Only          0.836   0.703       267      4243      4509
  4   Fixed Chunking + Hybrid + Rerank           0.966   0.714       239      2703      2942
  5   Recursive Chunking + Hybrid + Rerank       0.982   0.709       229      2832      3061  ← BEST
  6   Semantic Chunking + Hybrid + Rerank        0.832   0.692       239      3857      4096

  Best configuration : Config 5 — Recursive Chunking + Hybrid + Rerank
  Faithfulness       : 0.982
  Relevancy          : 0.709


In [14]:
# ── CELL 13: Show claim extraction examples for report ────────
print(f"\n{'='*60}")
print(f"  CLAIM EXTRACTION EXAMPLES (for report Section D)")
print(f"  Showing Config {best_id} — first 3 queries")
print(f"{'='*60}")

for ex in example_claims[best_id][:3]:
    print(f"\n  Question : {ex['question'][:80]}")
    print(f"  Answer   : {ex['answer'][:150]}...")
    print(f"  Sources  : {', '.join(ex['sources'][:2])}")
    print(f"  Claims extracted:")
    for c in ex["claims"]:
        icon = "✅" if c["supported"] else "❌"
        print(f"    {icon} {c['claim'][:80]}")
    print(f"  Faithfulness: {ex['faith']}  |  Relevancy: {ex['rel']}")


  CLAIM EXTRACTION EXAMPLES (for report Section D)
  Showing Config 5 — first 3 queries

  Question : How does the concept of Nested Learning relate to the idea of in-context learnin
  Answer   : The concept of Nested Learning (NL) relates to in-context learning (ICL) in that it provides a framework for understanding how models can adapt and le...
  Sources  : Nested Learning: The Illusion of Deep Learning Architectures, Nested Learning: The Illusion of Deep Learning Architectures
  Claims extracted:
    ✅ The concept of Nested Learning (NL) relates to in-context learning (ICL).
    ✅ NL provides a framework for understanding how models can adapt and learn from th
    ✅ NL involves a set of nested, multi-level, and/or parallel optimization problems.
    ✅ Existing deep learning methods compress their own context flow.
    ✅ In-context learning can naturally emerge in large models.
    ✅ ICL capabilities can be unified across different model architectures and objecti
    ✅ The connecti

In [15]:
# ── CELL 14: Save all results ─────────────────────────────────
with open("/kaggle/working/ablation_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

with open("/kaggle/working/example_claims.json", "w") as f:
    json.dump(example_claims, f, indent=2)

print(f"\n✓ Saved ablation_results.json")
print(f"✓ Saved example_claims.json")
print(f"\n  Use ablation_results.json for your report tables")
print(f"  Use example_claims.json for Section D claim extraction examples")
print(f"\n  Your best production config: {all_results[best_id]['label']}")
print(f"  Deploy this config to HF Spaces.")


✓ Saved ablation_results.json
✓ Saved example_claims.json

  Use ablation_results.json for your report tables
  Use example_claims.json for Section D claim extraction examples

  Your best production config: Recursive Chunking + Hybrid + Rerank
  Deploy this config to HF Spaces.


In [28]:
# ── CELL 15: Groq generation function (llama-3.1-8b-instant) ──────────────────
GROQ_GEN_MODEL = "llama-3.1-8b-instant"

def generate_answer_groq(query, chunks):
    """Generate answer using Groq llama-3.1-8b-instant (drop-in replacement for gpt-4o-mini)."""
    context = "\n\n---\n\n".join([
        f"[Source: {m.get('title','?')} ({m.get('year','?')})]\n{c}"
        for c, m in chunks
    ])
    prompt = f"""You are a research assistant specializing in machine learning.
Answer using ONLY the provided context from academic papers.
Do not fabricate facts or citations.
If context is insufficient, say so explicitly

Context:
{context}

Question: {query}
Answer:"""

    t0  = time.time()
    res = groq_client.chat.completions.create(
        model=GROQ_GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=512,
    )
    return res.choices[0].message.content, context, round(time.time() - t0, 3)

print(f"✓ generate_answer_groq() defined — model: {GROQ_GEN_MODEL}")


✓ generate_answer_groq() defined — model: llama-3.1-8b-instant


In [29]:
# ── CELL 16: Run ablation study with llama-3.1-8b-instant (Groq) ──────────────
#   Uses the exact same 6 configs × 30 questions as the original run.
#   Only the generation step is swapped to Groq; retrieval and evaluation
#   remain identical so results are directly comparable.

groq_results      = {}
groq_example_claims = {}

print(f"{'='*60}")
print(f"  GROQ ABLATION — 6 configurations × {len(golden)} questions")
print(f"  Generation : {GROQ_GEN_MODEL} (Groq)")
print(f"  Judge      : gpt-4o-mini (OpenAI)")
print(f"{'='*60}")

for config in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Config {config['id']}: {config['label']}")
    print(f"{'='*60}")

    faith_scores = []
    rel_scores   = []
    ret_times    = []
    gen_times    = []
    config_examples = []

    for i, item in enumerate(golden):
        q = item["question"]

        try:
            # Retrieve (same pipeline as before)
            chunks, ret_t = retrieve(
                q,
                strategy=config["strategy"],
                mode=config["mode"]
            )
            ret_times.append(ret_t)

            # Generate answer using llama-3.1-8b-instant
            answer, context, gen_t = generate_answer_groq(q, chunks)
            gen_times.append(gen_t)

            # Evaluate faithfulness (same judge: gpt-4o-mini)
            faith, claims = evaluate_faithfulness(answer, context)
            faith_scores.append(faith)

            # Evaluate relevancy (same embeddings: text-embedding-3-small)
            rel, gen_qs = evaluate_relevancy(q, answer)
            rel_scores.append(rel)

            # Save first 3 examples for report
            if i < 3:
                config_examples.append({
                    "question": q,
                    "answer":   answer,
                    "claims":   claims,
                    "faith":    faith,
                    "rel":      rel,
                    "sources":  [m.get("title", "?") for _, m in chunks],
                })

            print(f"  Q{i+1:02d} | Faith={faith:.3f} | Rel={rel:.3f} | "
                  f"Ret={ret_t}s | Gen={gen_t}s")

            time.sleep(0.5)

        except Exception as e:
            print(f"  Q{i+1:02d} ERROR: {e}")
            faith_scores.append(0.0)
            rel_scores.append(0.0)
            if ret_times: ret_times.append(ret_times[-1])
            if gen_times: gen_times.append(gen_times[-1])
            time.sleep(3)

    groq_results[config["id"]] = {
        "label":            config["label"],
        "strategy":         config["strategy"],
        "mode":             config["mode"],
        "avg_faithfulness": round(sum(faith_scores) / len(faith_scores), 3),
        "avg_relevancy":    round(sum(rel_scores)   / len(rel_scores),   3),
        "avg_ret_ms":       round(sum(ret_times) / len(ret_times) * 1000) if ret_times else 0,
        "avg_gen_ms":       round(sum(gen_times) / len(gen_times) * 1000) if gen_times else 0,
        "avg_total_ms":     round((sum(ret_times) + sum(gen_times)) / max(len(ret_times), 1) * 1000),
        "all_faith":        faith_scores,
        "all_rel":          rel_scores,
    }
    groq_example_claims[config["id"]] = config_examples

    print(f"\n  Config {config['id']} Summary (Groq):")
    print(f"    Avg Faithfulness : {groq_results[config['id']]['avg_faithfulness']}")
    print(f"    Avg Relevancy    : {groq_results[config['id']]['avg_relevancy']}")
    print(f"    Avg Ret Time     : {groq_results[config['id']]['avg_ret_ms']}ms")
    print(f"    Avg Gen Time     : {groq_results[config['id']]['avg_gen_ms']}ms")


  GROQ ABLATION — 6 configurations × 30 questions
  Generation : llama-3.1-8b-instant (Groq)
  Judge      : gpt-4o-mini (OpenAI)

  Config 1: Fixed Chunking + Semantic Only
  Q01 | Faith=1.000 | Rel=0.747 | Ret=0.307s | Gen=0.354s
  Q02 | Faith=1.000 | Rel=0.715 | Ret=0.259s | Gen=0.566s
  Q03 | Faith=1.000 | Rel=0.777 | Ret=0.198s | Gen=0.658s
  Q04 | Faith=0.750 | Rel=0.773 | Ret=0.181s | Gen=0.341s
  Q05 | Faith=1.000 | Rel=0.725 | Ret=0.22s | Gen=0.378s
  Q06 | Faith=1.000 | Rel=0.752 | Ret=0.194s | Gen=0.916s
  Q07 | Faith=1.000 | Rel=0.702 | Ret=0.193s | Gen=0.369s
  Q08 | Faith=1.000 | Rel=0.625 | Ret=0.191s | Gen=0.322s
  Q09 | Faith=0.909 | Rel=0.694 | Ret=0.213s | Gen=0.573s
  Q10 | Faith=0.875 | Rel=0.747 | Ret=0.17s | Gen=0.683s
  Q11 | Faith=1.000 | Rel=0.658 | Ret=0.212s | Gen=0.243s
  Q12 | Faith=1.000 | Rel=0.683 | Ret=0.308s | Gen=0.636s
  Q13 | Faith=0.933 | Rel=0.710 | Ret=0.173s | Gen=0.749s
  Q14 | Faith=1.000 | Rel=0.719 | Ret=0.244s | Gen=0.446s
  Q15 | Faith=1.0

In [38]:
# ── CELL 17: Side-by-side comparison — gpt-4o-mini vs llama-3.1-8b-instant ────

print(f"\n{'='*100}")
print(f"  MODEL COMPARISON: gpt-4o-mini  vs  {GROQ_GEN_MODEL}")
print(f"{'='*100}")
print(f"  {'#':<3} {'Configuration':<40} "
      f"{'GPT Faith':>9} {'GPT Rel':>7} {'GPT Gen(ms)':>11} "
      f"{'Groq Faith':>10} {'Groq Rel':>8} {'Groq Gen(ms)':>12} "
      f"{'ΔFaith':>7} {'ΔRel':>7}")
print(f"  {'-'*100}")

for cid in all_results:
    gpt  = all_results[cid]
    groq = groq_results[cid]
    d_faith = round(groq["avg_faithfulness"] - gpt["avg_faithfulness"], 3)
    d_rel   = round(groq["avg_relevancy"]    - gpt["avg_relevancy"],    3)
    d_faith_str = f"+{d_faith}" if d_faith >= 0 else str(d_faith)
    d_rel_str   = f"+{d_rel}"   if d_rel   >= 0 else str(d_rel)
    print(f"  {cid:<3} {gpt['label']:<40} "
          f"{gpt['avg_faithfulness']:>9} {gpt['avg_relevancy']:>7} {gpt['avg_gen_ms']:>11} "
          f"{groq['avg_faithfulness']:>10} {groq['avg_relevancy']:>8} {groq['avg_gen_ms']:>12} "
          f"{d_faith_str:>7} {d_rel_str:>7}")

# Overall averages
avg_gpt_f    = round(sum(r["avg_faithfulness"] for r in all_results.values())  / len(all_results),  3)
avg_gpt_r    = round(sum(r["avg_relevancy"]    for r in all_results.values())  / len(all_results),  3)
avg_groq_f   = round(sum(r["avg_faithfulness"] for r in groq_results.values()) / len(groq_results), 3)
avg_groq_r   = round(sum(r["avg_relevancy"]    for r in groq_results.values()) / len(groq_results), 3)
avg_gpt_gen  = round(sum(r["avg_gen_ms"]       for r in all_results.values())  / len(all_results))
avg_groq_gen = round(sum(r["avg_gen_ms"]       for r in groq_results.values()) / len(groq_results))

print(f"\n  {'OVERALL AVERAGE':<44} "
      f"{avg_gpt_f:>9} {avg_gpt_r:>7} {avg_gpt_gen:>11} "
      f"{avg_groq_f:>10} {avg_groq_r:>8} {avg_groq_gen:>12} "
      f"{round(avg_groq_f - avg_gpt_f, 3):>+7} {round(avg_groq_r - avg_gpt_r, 3):>+7}")

# Speed comparison
speed_ratio = round(avg_gpt_gen / avg_groq_gen, 1) if avg_groq_gen else float("inf")
print(f"\n  Generation speed: Groq is ~{speed_ratio}x faster than gpt-4o-mini on average")

# Best config per model
best_gpt_id  = max(all_results,  key=lambda x: all_results[x]["avg_faithfulness"]  + all_results[x]["avg_relevancy"])
best_groq_id = max(groq_results, key=lambda x: groq_results[x]["avg_faithfulness"] + groq_results[x]["avg_relevancy"])
print(f"\n  Best config (gpt-4o-mini)           : Config {best_gpt_id}  — {all_results[best_gpt_id]['label']}")
print(f"  Best config ({GROQ_GEN_MODEL}) : Config {best_groq_id}  — {groq_results[best_groq_id]['label']}")



  MODEL COMPARISON: gpt-4o-mini  vs  llama-3.1-8b-instant
  #   Configuration                            GPT Faith GPT Rel GPT Gen(ms) Groq Faith Groq Rel Groq Gen(ms)  ΔFaith    ΔRel
  ----------------------------------------------------------------------------------------------------
  1   Fixed Chunking + Semantic Only               0.925   0.688        2721      0.944    0.703          493  +0.019  +0.015
  2   Recursive Chunking + Semantic Only           0.964   0.696        4020      0.941    0.702          403  -0.023  +0.006
  3   Semantic Chunking + Semantic Only            0.836   0.703        4243      0.744    0.696          604  -0.092  -0.007
  4   Fixed Chunking + Hybrid + Rerank             0.966   0.714        2703      0.965    0.712          420  -0.001  -0.002
  5   Recursive Chunking + Hybrid + Rerank         0.982   0.709        2832       0.97    0.717          393  -0.012  +0.008
  6   Semantic Chunking + Hybrid + Rerank          0.832   0.692        3857      

In [39]:
# ── CELL 18: Save Groq results ────────────────────────────────────────────────
with open("/kaggle/working/groq_ablation_results.json", "w") as f:
    json.dump(groq_results, f, indent=2)

with open("/kaggle/working/groq_example_claims.json", "w") as f:
    json.dump(groq_example_claims, f, indent=2)

# Combined comparison dict for easy report ingestion
comparison = {
    "gpt-4o-mini":       all_results,
    GROQ_GEN_MODEL:      groq_results,
}
with open("/kaggle/working/model_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

print("✓ groq_ablation_results.json saved")
print("✓ groq_example_claims.json saved")
print("✓ model_comparison.json saved  ← use this for report tables")


✓ groq_ablation_results.json saved
✓ groq_example_claims.json saved
✓ model_comparison.json saved  ← use this for report tables


In [53]:
# ── TEST A NEW QUESTION ────────────────────────────────────────────────────────
test_question = "How does Prompt Injection Mitigation use the idea of nested learning"   # ← change this

# Config to test with (5 = best: Recursive + Hybrid + Rerank)
TEST_STRATEGY = "B"
TEST_MODE     = "hybrid"

print(f"Question : {test_question}")
print(f"Strategy : {TEST_STRATEGY} | Mode: {TEST_MODE}")
print("="*60)

# Retrieve
chunks, ret_t = retrieve(test_question, strategy=TEST_STRATEGY, mode=TEST_MODE)

print(f"\n📄 Retrieved Chunks ({ret_t}s):")
for i, (chunk, meta) in enumerate(chunks, 1):
    print(f"\n  Chunk {i} | {meta.get('title','?')} ({meta.get('year','?')})")
    print(f"  {chunk[:200]}...")

# Generate with GPT
answer_gpt, context, gen_t_gpt = generate_answer(test_question, chunks)
print(f"\n🤖 GPT-4o-mini Answer ({gen_t_gpt}s):\n{answer_gpt}")

# Generate with Groq
answer_groq, _, gen_t_groq = generate_answer_groq(test_question, chunks)
print(f"\n⚡ Llama-3.1-8b Answer ({gen_t_groq}s):\n{answer_groq}")

# Evaluate both
faith_gpt, _ = evaluate_faithfulness(answer_gpt, context)
rel_gpt, _   = evaluate_relevancy(test_question, answer_gpt)

faith_groq, _ = evaluate_faithfulness(answer_groq, context)
rel_groq, _   = evaluate_relevancy(test_question, answer_groq)

print(f"\n{'='*60}")
print(f"  SCORES")
print(f"{'='*60}")
print(f"  {'Model':<25} {'Faithfulness':>12} {'Relevancy':>10} {'Gen(ms)':>9}")
print(f"  {'-'*60}")
print(f"  {'gpt-4o-mini':<25} {faith_gpt:>12} {rel_gpt:>10} {round(gen_t_gpt*1000):>9}")
print(f"  {'llama-3.1-8b-instant':<25} {faith_groq:>12} {rel_groq:>10} {round(gen_t_groq*1000):>9}")

Question : How does Prompt Injection Mitigation use the idea of nested learning
Strategy : B | Mode: hybrid

📄 Retrieved Chunks (0.525s):

  Chunk 1 | Prompt Injection Mitigation with Agentic AI, Nested Learning, and AI Sustainability (2026)
  from Nested Learning can be translated into an operational prompt injection defence.
Our prior work [11] established the baseline multi-agent architecture on 500 synthetic prompts
using a four-metric ...

  Chunk 2 | Prompt Injection Mitigation with Agentic AI, Nested Learning, and AI Sustainability (2026)
  The remainder of this paper develops these ideas in detail. After reviewing related work
in prompt injection and multi-agent security, the text introduces the Nested Learning archi-
tecture and explai...

  Chunk 3 | Prompt Injection Mitigation with Agentic AI, Nested Learning, and AI Sustainability (2026)
  Prompt Injection Mitigation with Agentic AI, Nested Learning,
and AI Sustainability via Semantic Caching
Diego Gosmar
Head of AI, Tesisq

In [54]:
# ── CELL 19: Run Config 5 with alternative judge (llama-3.3-70b-versatile) ────
#   Generation : gpt-4o-mini (same as original)
#   Judge      : llama-3.3-70b-versatile (Groq) ← only change
#   Config     : 5 — Recursive Chunking + Hybrid + Rerank (best)

JUDGE_MODEL  = "llama-3.1-8b-instant"
BEST_CONFIG  = next(c for c in CONFIGS if c["id"] == 5)

def llm_judge_groq(prompt, max_tokens=400):
    """Same as llm_judge() but uses Groq llama-3.3-70b-versatile instead of gpt-4o-mini."""
    r = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content.strip()


def evaluate_faithfulness_groq(answer, context):
    """Faithfulness evaluation using Groq judge."""
    claims_raw = llm_judge_groq(
        f"""Extract all distinct factual claims from this answer.
Return ONLY a numbered list, one claim per line. No preamble or explanation.

Answer: {answer}""", max_tokens=400)

    claims = []
    for line in claims_raw.strip().split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            claim = line.split(". ", 1)[-1].strip("- ").strip()
            if len(claim) > 15:
                claims.append(claim)

    if not claims:
        return 0.0, []

    verified, results = 0, []
    for claim in claims:
        verdict = llm_judge_groq(
            f"""Context from academic papers:
{context[:2500]}

Is this claim directly supported by the context above?
Answer with YES or NO only. No explanation.

Claim: {claim}""", max_tokens=5)

        supported = verdict.upper().startswith("YES")
        verified += int(supported)
        results.append({"claim": claim, "supported": supported})
        time.sleep(0.3)

    return round(verified / len(claims), 3), results


def evaluate_relevancy_groq(query, answer):
    """Relevancy evaluation using Groq judge to generate questions."""
    gen_qs_raw = llm_judge_groq(
        f"""Generate exactly 3 questions that this answer is responding to.
Return ONLY the 3 questions, one per line. No numbering, no explanation.

Answer: {answer}""", max_tokens=200)

    gen_qs = [q.strip() for q in gen_qs_raw.strip().split("\n")
              if q.strip() and len(q.strip()) > 10][:3]

    if not gen_qs:
        return 0.0, []

    # Embeddings still use text-embedding-3-small (unchanged)
    orig_emb = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    gen_embs = []
    for q in gen_qs:
        emb = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=q
        ).data[0].embedding
        gen_embs.append(emb)

    sims      = cosine_similarity([orig_emb], gen_embs)[0]
    avg_score = round(float(np.mean(sims)), 3)

    return avg_score, list(zip(gen_qs, [round(float(s), 3) for s in sims]))


# ── Run ───────────────────────────────────────────────────────────────────────
print(f"{'='*60}")
print(f"  Config {BEST_CONFIG['id']}: {BEST_CONFIG['label']}")
print(f"  Generation : gpt-4o-mini")
print(f"  Judge      : {JUDGE_MODEL} (Groq)")
print(f"{'='*60}")

groq_judge_results = {"faith": [], "rel": [], "ret": [], "gen": []}

for i, item in enumerate(golden):
    q = item["question"]

    try:
        # Retrieve (same pipeline)
        chunks, ret_t = retrieve(q, strategy=BEST_CONFIG["strategy"],
                                    mode=BEST_CONFIG["mode"])
        groq_judge_results["ret"].append(ret_t)

        # Generate with gpt-4o-mini (unchanged)
        answer, context, gen_t = generate_answer(q, chunks)
        groq_judge_results["gen"].append(gen_t)

        # Judge with llama-3.3-70b-versatile
        faith, _ = evaluate_faithfulness_groq(answer, context)
        rel,   _ = evaluate_relevancy_groq(q, answer)

        groq_judge_results["faith"].append(faith)
        groq_judge_results["rel"].append(rel)

        print(f"  Q{i+1:02d} | Faith={faith:.3f} | Rel={rel:.3f} | "
              f"Ret={ret_t}s | Gen={gen_t}s")

        time.sleep(0.5)

    except Exception as e:
        print(f"  Q{i+1:02d} ERROR: {e}")
        groq_judge_results["faith"].append(0.0)
        groq_judge_results["rel"].append(0.0)
        time.sleep(3)

# ── Comparison: original judge vs new judge ────────────────────────────────────
orig = all_results[5]   # Config 5 original scores

avg_faith_new = round(sum(groq_judge_results["faith"]) / len(groq_judge_results["faith"]), 3)
avg_rel_new   = round(sum(groq_judge_results["rel"])   / len(groq_judge_results["rel"]),   3)

print(f"\n{'='*60}")
print(f"  JUDGE COMPARISON — Config 5: {BEST_CONFIG['label']}")
print(f"  Generation model is gpt-4o-mini in both cases")
print(f"{'='*60}")
print(f"  {'Judge':<30} {'Faithfulness':>12} {'Relevancy':>10}")
print(f"  {'-'*55}")
print(f"  {'gpt-4o-mini (original)':<30} {orig['avg_faithfulness']:>12} {orig['avg_relevancy']:>10}")
print(f"  {JUDGE_MODEL:<30} {avg_faith_new:>12} {avg_rel_new:>10}")
print(f"  {'-'*55}")
print(f"  {'Delta':<30} {round(avg_faith_new - orig['avg_faithfulness'], 3):>+12} "
      f"{round(avg_rel_new - orig['avg_relevancy'], 3):>+10}")
print(f"\n  {'↑ stricter judge' if avg_faith_new < orig['avg_faithfulness'] else '↓ more lenient judge'}")

  Config 5: Recursive Chunking + Hybrid + Rerank
  Generation : gpt-4o-mini
  Judge      : llama-3.1-8b-instant (Groq)
  Q01 | Faith=1.000 | Rel=0.767 | Ret=0.495s | Gen=4.482s
  Q02 | Faith=0.800 | Rel=0.654 | Ret=0.203s | Gen=3.499s
  Q03 | Faith=0.889 | Rel=0.678 | Ret=0.32s | Gen=3.128s
  Q04 | Faith=1.000 | Rel=0.817 | Ret=0.21s | Gen=1.658s
  Q05 | Faith=0.778 | Rel=0.826 | Ret=0.258s | Gen=3.908s
  Q06 | Faith=0.833 | Rel=0.871 | Ret=0.222s | Gen=2.735s
  Q07 | Faith=1.000 | Rel=0.650 | Ret=0.244s | Gen=2.394s
  Q08 | Faith=0.909 | Rel=0.499 | Ret=0.259s | Gen=3.448s
  Q09 | Faith=1.000 | Rel=0.679 | Ret=0.313s | Gen=3.49s
  Q10 | Faith=0.800 | Rel=0.606 | Ret=0.327s | Gen=3.372s
  Q11 | Faith=1.000 | Rel=0.751 | Ret=0.346s | Gen=1.605s
  Q12 | Faith=1.000 | Rel=0.681 | Ret=0.21s | Gen=3.49s
  Q13 | Faith=1.000 | Rel=0.725 | Ret=0.204s | Gen=4.097s
  Q14 | Faith=0.625 | Rel=0.684 | Ret=0.771s | Gen=3.011s
  Q15 | Faith=1.000 | Rel=0.570 | Ret=0.193s | Gen=2.709s
  Q16 | Faith=1.

In [55]:
# ── CELL 20: Random questions | High temperature | Both generators | GPT judge ──
import random

# ── Settings ──────────────────────────────────────────────────────────────────
N_QUESTIONS  = 10          # ← change how many random questions to sample
TEMPERATURE  = 0.7         # ← increased from 0.1
BEST_CONFIG  = next(c for c in CONFIGS if c["id"] == 5)

# Random sample from golden set
random.seed(42)            # ← change or remove seed for different questions each run
sampled = random.sample(golden, N_QUESTIONS)

print(f"{'='*60}")
print(f"  Random Subset Test")
print(f"  Config     : {BEST_CONFIG['id']} — {BEST_CONFIG['label']}")
print(f"  Questions  : {N_QUESTIONS} random from {len(golden)}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Generators : gpt-4o-mini + llama-3.1-8b-instant")
print(f"  Judge      : gpt-4o-mini")
print(f"{'='*60}")

# ── High temperature generation functions ──────────────────────────────────────
def generate_answer_gpt_hot(query, chunks):
    context = "\n\n---\n\n".join([
        f"[Source: {m.get('title','?')} ({m.get('year','?')})]  \n{c}"
        for c, m in chunks
    ])
    prompt = f"""You are a research assistant specializing in machine learning.
Answer using ONLY the provided context from academic papers.
Do not fabricate facts or citations.
If context is insufficient, say so explicitly.

Context:
{context}

Question: {query}
Answer:"""

    t0  = time.time()
    res = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=512,
    )
    return res.choices[0].message.content, context, round(time.time() - t0, 3)


def generate_answer_groq_hot(query, chunks):
    context = "\n\n---\n\n".join([
        f"[Source: {m.get('title','?')} ({m.get('year','?')})]  \n{c}"
        for c, m in chunks
    ])
    prompt = f"""You are a research assistant specializing in machine learning.
Answer using ONLY the provided context from academic papers.
Do not fabricate facts or citations.
If context is insufficient, say so explicitly.

Context:
{context}

Question: {query}
Answer:"""

    t0  = time.time()
    res = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=512,
    )
    return res.choices[0].message.content, context, round(time.time() - t0, 3)


# ── Run ────────────────────────────────────────────────────────────────────────
gpt_faith,  gpt_rel,  gpt_gen  = [], [], []
groq_faith, groq_rel, groq_gen = [], [], []

for i, item in enumerate(sampled):
    q = item["question"]
    print(f"\n  Q{i+1:02d}: {q[:70]}")
    print(f"  {'-'*55}")

    try:
        # Retrieve once — shared by both generators
        chunks, ret_t = retrieve(q, strategy=BEST_CONFIG["strategy"],
                                    mode=BEST_CONFIG["mode"])

        # GPT generation + evaluation
        ans_gpt,  ctx_gpt,  gen_t_gpt  = generate_answer_gpt_hot(q, chunks)
        faith_gpt,  _ = evaluate_faithfulness(ans_gpt,  ctx_gpt)
        rel_gpt,    _ = evaluate_relevancy(q, ans_gpt)
        gpt_faith.append(faith_gpt)
        gpt_rel.append(rel_gpt)
        gpt_gen.append(gen_t_gpt)

        # Groq generation + evaluation
        ans_groq, ctx_groq, gen_t_groq = generate_answer_groq_hot(q, chunks)
        faith_groq, _ = evaluate_faithfulness(ans_groq, ctx_groq)
        rel_groq,   _ = evaluate_relevancy(q, ans_groq)
        groq_faith.append(faith_groq)
        groq_rel.append(rel_groq)
        groq_gen.append(gen_t_groq)

        print(f"  GPT  | Faith={faith_gpt:.3f} | Rel={rel_gpt:.3f} | Gen={gen_t_gpt}s")
        print(f"  Groq | Faith={faith_groq:.3f} | Rel={rel_groq:.3f} | Gen={gen_t_groq}s")

        time.sleep(0.5)

    except Exception as e:
        print(f"  ERROR: {e}")
        gpt_faith.append(0.0);  gpt_rel.append(0.0)
        groq_faith.append(0.0); groq_rel.append(0.0)
        time.sleep(3)

# ── Summary ────────────────────────────────────────────────────────────────────
avg = lambda lst: round(sum(lst) / len(lst), 3)

print(f"\n{'='*60}")
print(f"  SUMMARY — Temperature={TEMPERATURE} vs Original Temperature=0.1")
print(f"{'='*60}")
print(f"  {'Model':<25} {'Faith':>7} {'Rel':>7} {'Gen(ms)':>9}")
print(f"  {'-'*55}")
print(f"  {'gpt-4o-mini (hot)':<25} {avg(gpt_faith):>7} {avg(gpt_rel):>7} {round(avg(gpt_gen)*1000):>9}")
print(f"  {'llama-3.1-8b (hot)':<25} {avg(groq_faith):>7} {avg(groq_rel):>7} {round(avg(groq_gen)*1000):>9}")
print(f"  {'-'*55}")

# Compare against original Config 5 scores (temperature=0.1)
orig_gpt  = all_results[5]
orig_groq = groq_results[5]
print(f"  {'gpt-4o-mini (orig 0.1)':<25} {orig_gpt['avg_faithfulness']:>7} {orig_gpt['avg_relevancy']:>7} {orig_gpt['avg_gen_ms']:>9}")
print(f"  {'llama-3.1-8b (orig 0.1)':<25} {orig_groq['avg_faithfulness']:>7} {orig_groq['avg_relevancy']:>7} {orig_groq['avg_gen_ms']:>9}")
print(f"\n  Faith drop (GPT)  : {round(avg(gpt_faith)  - orig_gpt['avg_faithfulness'],  3):>+.3f}")
print(f"  Faith drop (Groq) : {round(avg(groq_faith) - orig_groq['avg_faithfulness'], 3):>+.3f}")
print(f"\n  ℹ️  Negative delta = higher temperature hurt faithfulness as expected")

  Random Subset Test
  Config     : 5 — Recursive Chunking + Hybrid + Rerank
  Questions  : 10 random from 30
  Temperature: 0.7
  Generators : gpt-4o-mini + llama-3.1-8b-instant
  Judge      : gpt-4o-mini

  Q01: What limitations do existing approaches to extending the effective con
  -------------------------------------------------------
  GPT  | Faith=1.000 | Rel=0.894 | Gen=5.116s
  Groq | Faith=1.000 | Rel=0.849 | Gen=0.369s

  Q02: What is the role of the programmer network in Fast Weight Programmers 
  -------------------------------------------------------
  GPT  | Faith=1.000 | Rel=0.731 | Gen=1.787s
  Groq | Faith=1.000 | Rel=0.541 | Gen=0.247s

  Q03: How does the concept of Nested Learning relate to the idea of in-conte
  -------------------------------------------------------
  GPT  | Faith=1.000 | Rel=0.710 | Gen=4.52s
  Groq | Faith=1.000 | Rel=0.801 | Gen=0.47s

  Q04: What is the significance of the fifth metric, Observability Score Rati
  ----------------------------

In [56]:
# -- CELL 21: Prompt Engineering Ablation -- Config 5 (Best) ------------------
#
#  What changes here (relative to all prior cells):
#  +-----------------------+--------------------------------------------------+
#  | Component             | What we vary                                     |
#  +-----------------------+--------------------------------------------------+
#  | Generation prompt     | Baseline -> CoT + citation anchoring             |
#  | Faithfulness judge    | Baseline -> Rubric + step-by-step verification   |
#  | Relevancy judge       | Baseline -> Aspect-decomposed scoring rubric     |
#  +-----------------------+--------------------------------------------------+
#
#  We run 4 prompt configurations on Config 5 (Recursive + Hybrid + Rerank):
#
#  [P1] original_gen + original_judge  <- baseline (re-used from all_results[5])
#  [P2] improved_gen + original_judge  <- isolate generation prompt effect
#  [P3] original_gen + improved_judge  <- isolate judge prompt effect
#  [P4] improved_gen + improved_judge  <- full improvement
#
#  Runs on a fixed 15-question subset for cost efficiency.
#  All retrieval stays identical (Config 5: strategy B, hybrid mode).
# ------------------------------------------------------------------------------

import random, time, json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

PROMPT_SAMPLE_SIZE = 15
PROMPT_TEMPERATURE = 0.1          # keep consistent with original ablation
BEST_CFG           = next(c for c in CONFIGS if c["id"] == 5)

random.seed(7)
prompt_sample = random.sample(golden, PROMPT_SAMPLE_SIZE)

# ---- 1. GENERATION PROMPTS ---------------------------------------------------

def build_context(chunks):
    return "\n\n---\n\n".join([
        f"[Source {i+1}: {m.get('title','?')} ({m.get('year','?')})]\n{c}"
        for i, (c, m) in enumerate(chunks)
    ])


def gen_original(query, chunks):
    """Original generation prompt (identical to generate_answer in Cell 7)."""
    context = "\n\n---\n\n".join([
        f"[Source: {m.get('title','?')} ({m.get('year','?')})]\n{c}"
        for c, m in chunks
    ])
    prompt = (
        "You are a research assistant specializing in machine learning.\n"
        "Answer using ONLY the provided context from academic papers.\n"
        "Do not fabricate facts or citations.\n"
        "If context is insufficient, say so explicitly.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    t0  = time.time()
    res = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=PROMPT_TEMPERATURE,
        max_tokens=512,
    )
    return res.choices[0].message.content.strip(), context, round(time.time()-t0, 3)


def gen_improved(query, chunks):
    """
    Improved generation prompt:
    - System/user role separation for sharper instruction following
    - Explicit chain-of-thought step before the final answer
    - Mandatory inline source citation using (Source N) notation
    - Structured output: [Reasoning] block + [Answer] block
    - Explicit hedging instruction when evidence is partial
    """
    context = build_context(chunks)
    system_msg = (
        "You are an expert research assistant who answers questions strictly "
        "from provided academic-paper excerpts. "
        "You NEVER introduce facts, numbers, or claims not present in the excerpts. "
        "When you use a fact, cite its source inline as (Source N). "
        "If the excerpts do not contain enough information, state that clearly."
    )
    user_msg = (
        f"Below are numbered excerpts from academic papers. Read them carefully.\n\n"
        f"{context}\n\n"
        "---\n"
        f"QUESTION: {query}\n\n"
        "Think step by step:\n"
        "1. Identify which sources are relevant to the question.\n"
        "2. Extract the key facts from those sources.\n"
        "3. Synthesise a concise, well-cited answer.\n\n"
        "Respond in this exact format:\n\n"
        "[Reasoning]\n"
        "<2-4 sentences identifying the relevant sources and their key points>\n\n"
        "[Answer]\n"
        "<Your final answer with inline citations like (Source 1), (Source 2), etc.>"
    )
    t0  = time.time()
    res = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ],
        temperature=PROMPT_TEMPERATURE,
        max_tokens=600,
    )
    raw = res.choices[0].message.content.strip()
    # Extract only the [Answer] block for downstream evaluation
    if "[Answer]" in raw:
        answer = raw.split("[Answer]", 1)[1].strip()
    else:
        answer = raw
    return answer, context, round(time.time()-t0, 3)


# ---- 2. JUDGE PROMPTS --------------------------------------------------------

# 2a. Faithfulness judges

def faithfulness_original(answer, context):
    """Original faithfulness judge (identical to evaluate_faithfulness in Cell 8)."""
    claims_raw = llm_judge(
        "Extract all distinct factual claims from this answer.\n"
        "Return ONLY a numbered list, one claim per line. No preamble or explanation.\n\n"
        f"Answer: {answer}",
        max_tokens=400
    )
    claims = []
    for line in claims_raw.strip().split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            claim = line.split(". ", 1)[-1].strip("- ").strip()
            if len(claim) > 15:
                claims.append(claim)
    if not claims:
        return 0.0, []
    verified, results = 0, []
    for claim in claims:
        verdict = llm_judge(
            f"Context from academic papers:\n{context[:2500]}\n\n"
            "Is this claim directly supported by the context above?\n"
            "Answer with YES or NO only. No explanation.\n\n"
            f"Claim: {claim}",
            max_tokens=5
        )
        supported = verdict.upper().startswith("YES")
        verified += int(supported)
        results.append({"claim": claim, "supported": supported})
        time.sleep(0.3)
    return round(verified / len(claims), 3), results


def faithfulness_improved(answer, context):
    """
    Improved faithfulness judge:
    - Extraction step guides atomic decomposition with explicit scope rules
    - Verification uses a 3-level rubric: SUPPORTED / PARTIAL / NOT_SUPPORTED
      mapped to 1.0 / 0.5 / 0.0 credit (finer-grained than binary YES/NO)
    - Larger context window (3500 chars vs 2500)
    """
    # Step 1: Extract atomic claims
    claims_raw = llm_judge(
        "You are evaluating a RAG system answer for factual faithfulness.\n\n"
        "Task: Extract every distinct, atomic factual claim from the answer below.\n"
        "Rules:\n"
        "- One claim per line, numbered.\n"
        "- Each claim must be a single verifiable statement (not compound).\n"
        "- Include numerical values, comparisons, and named entities as separate claims.\n"
        "- Exclude vague statements like 'the paper discusses...' - only concrete facts.\n"
        "- No preamble, no explanation - just the numbered list.\n\n"
        f"Answer to decompose:\n{answer}",
        max_tokens=500
    )
    claims = []
    for line in claims_raw.strip().split("\n"):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith("-")):
            claim = line.split(". ", 1)[-1].strip("- ").strip()
            if len(claim) > 15:
                claims.append(claim)
    if not claims:
        return 0.0, []

    # Step 2: Verify each claim with 3-level rubric
    verified_score, results = 0.0, []
    for claim in claims:
        verdict = llm_judge(
            "You are a strict factual verifier for academic RAG systems.\n\n"
            f"Context (excerpts from academic papers):\n{context[:3500]}\n\n"
            f'Claim to verify: "{claim}"\n\n'
            "Evaluate whether the claim is supported by the context above.\n"
            "Use EXACTLY one of these three verdicts:\n"
            "  SUPPORTED     - the context explicitly states or clearly implies this claim\n"
            "  PARTIAL       - the context partially supports it but key details are missing\n"
            "  NOT_SUPPORTED - the context does not support this claim or contradicts it\n\n"
            "Respond with a single word: SUPPORTED, PARTIAL, or NOT_SUPPORTED",
            max_tokens=10
        )
        v = verdict.strip().upper()
        if "SUPPORTED" in v and "NOT" not in v and "PARTIAL" not in v:
            credit = 1.0
        elif "PARTIAL" in v:
            credit = 0.5
        else:
            credit = 0.0
        verified_score += credit
        results.append({"claim": claim, "supported": credit >= 0.5,
                         "credit": credit, "verdict": v})
        time.sleep(0.3)
    return round(verified_score / len(claims), 3), results


# 2b. Relevancy judges

def relevancy_original(query, answer):
    """Original relevancy judge (identical to evaluate_relevancy in Cell 8)."""
    gen_qs_raw = llm_judge(
        "Generate exactly 3 questions that this answer is responding to.\n"
        "Return ONLY the 3 questions, one per line. No numbering, no explanation.\n\n"
        f"Answer: {answer}",
        max_tokens=200
    )
    gen_qs = [q.strip() for q in gen_qs_raw.strip().split("\n")
              if q.strip() and len(q.strip()) > 10][:3]
    if not gen_qs:
        return 0.0, []
    orig_emb = openai_client.embeddings.create(
        model="text-embedding-3-small", input=query
    ).data[0].embedding
    gen_embs = []
    for q in gen_qs:
        emb = openai_client.embeddings.create(
            model="text-embedding-3-small", input=q
        ).data[0].embedding
        gen_embs.append(emb)
    sims      = cosine_similarity([orig_emb], gen_embs)[0]
    avg_score = round(float(np.mean(sims)), 3)
    return avg_score, list(zip(gen_qs, [round(float(s), 3) for s in sims]))


def relevancy_improved(query, answer):
    """
    Improved relevancy judge using aspect-decomposed scoring:
    - LLM scores 4 explicit relevancy aspects (0.0-1.0 each) instead of
      a proxy question-generation + cosine-similarity trick
    - Aspects: topic_alignment, specificity_match, scope_match, completeness
    - Final score = weighted mean of 4 aspect scores
    - Falls back to embedding similarity if JSON parse fails
    """
    aspect_prompt = (
        "You are evaluating how well a RAG system answer addresses a user question.\n\n"
        f"QUESTION: {query}\n\n"
        f"ANSWER: {answer}\n\n"
        "Score the answer on the following 4 aspects. For each, give a score from\n"
        "0.0 to 1.0 (step 0.25) and a one-sentence justification.\n\n"
        "Aspects:\n"
        "1. topic_alignment    - Does the answer address the same topic as the question?\n"
        "2. specificity_match  - Does the answer provide the level of detail asked for?\n"
        "3. scope_match        - Does the answer stay within scope (no irrelevant tangents)?\n"
        "4. completeness       - Does the answer address all parts of the question?\n\n"
        "Respond ONLY with valid JSON (no markdown, no code fences):\n"
        '{"topic_alignment":   {"score": <0-1>, "reason": "<sentence>"},\n'
        ' "specificity_match": {"score": <0-1>, "reason": "<sentence>"},\n'
        ' "scope_match":       {"score": <0-1>, "reason": "<sentence>"},\n'
        ' "completeness":      {"score": <0-1>, "reason": "<sentence>"}}'
    )
    raw = llm_judge(aspect_prompt, max_tokens=350)
    try:
        clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        scores_dict = json.loads(clean)
        weights = {"topic_alignment": 0.35, "specificity_match": 0.25,
                   "scope_match": 0.20,     "completeness": 0.20}
        weighted = sum(
            scores_dict[k]["score"] * w
            for k, w in weights.items()
            if k in scores_dict
        )
        aspects = [(k, scores_dict[k]["score"], scores_dict[k]["reason"])
                   for k in weights if k in scores_dict]
        return round(weighted, 3), aspects
    except Exception:
        # Fallback: use original embedding method
        return relevancy_original(query, answer)


# ---- 3. RUNNER: 3 new prompt configurations x 15 questions ------------------

PROMPT_CONFIGS = [
    {"id": "P2", "label": "Improved Gen   + Original Judge",
     "gen_fn": gen_improved,   "faith_fn": faithfulness_original, "rel_fn": relevancy_original},
    {"id": "P3", "label": "Original Gen   + Improved Judge",
     "gen_fn": gen_original,   "faith_fn": faithfulness_improved, "rel_fn": relevancy_improved},
    {"id": "P4", "label": "Improved Gen   + Improved Judge",
     "gen_fn": gen_improved,   "faith_fn": faithfulness_improved, "rel_fn": relevancy_improved},
]

# P1 baseline comes from all_results[5] -- no need to re-run API calls.
prompt_results = {}

for pcfg in PROMPT_CONFIGS:
    print(f"\n{'='*65}")
    print(f"  {pcfg['id']}: {pcfg['label']}")
    print(f"{'='*65}")
    faith_scores, rel_scores, gen_times = [], [], []

    for i, item in enumerate(prompt_sample):
        q = item["question"]
        try:
            chunks, _ = retrieve(q, strategy=BEST_CFG["strategy"],
                                    mode=BEST_CFG["mode"])
            answer, context, gen_t = pcfg["gen_fn"](q, chunks)
            gen_times.append(gen_t)
            faith, _ = pcfg["faith_fn"](answer, context)
            faith_scores.append(faith)
            rel, _ = pcfg["rel_fn"](q, answer)
            rel_scores.append(rel)
            print(f"  Q{i+1:02d} | Faith={faith:.3f} | Rel={rel:.3f} | Gen={gen_t}s")
            time.sleep(0.5)
        except Exception as e:
            print(f"  Q{i+1:02d} ERROR: {e}")
            faith_scores.append(0.0)
            rel_scores.append(0.0)
            time.sleep(3)

    prompt_results[pcfg["id"]] = {
        "label":      pcfg["label"],
        "avg_faith":  round(sum(faith_scores) / len(faith_scores), 3),
        "avg_rel":    round(sum(rel_scores)   / len(rel_scores),   3),
        "avg_gen_ms": round(sum(gen_times) / len(gen_times) * 1000) if gen_times else 0,
        "all_faith":  faith_scores,
        "all_rel":    rel_scores,
    }
    print(f"\n  Summary -- {pcfg['id']}:")
    print(f"    Avg Faithfulness : {prompt_results[pcfg['id']]['avg_faith']}")
    print(f"    Avg Relevancy    : {prompt_results[pcfg['id']]['avg_rel']}")
    print(f"    Avg Gen Time     : {prompt_results[pcfg['id']]['avg_gen_ms']}ms")


# ---- 4. COMPARISON TABLE ----------------------------------------------------

# P1 baseline -- re-use first PROMPT_SAMPLE_SIZE scores from original run
p1_faith_all = all_results[5]["all_faith"]
p1_rel_all   = all_results[5]["all_rel"]
p1_faith_sub = round(sum(p1_faith_all[:PROMPT_SAMPLE_SIZE]) / PROMPT_SAMPLE_SIZE, 3)
p1_rel_sub   = round(sum(p1_rel_all[:PROMPT_SAMPLE_SIZE])   / PROMPT_SAMPLE_SIZE, 3)
p1_gen_ms    = all_results[5]["avg_gen_ms"]

prompt_results["P1"] = {
    "label":      "Original Gen   + Original Judge  (baseline)",
    "avg_faith":  p1_faith_sub,
    "avg_rel":    p1_rel_sub,
    "avg_gen_ms": p1_gen_ms,
}

print(f"\n{'='*80}")
print(f"  PROMPT ENGINEERING COMPARISON  -- Config 5: {BEST_CFG['label']}")
print(f"  Base model: gpt-4o-mini  |  Questions: {PROMPT_SAMPLE_SIZE} (random subset, seed=7)")
print(f"{'='*80}")
print(f"  {'ID':<4} {'Configuration':<44} {'Faith':>7} {'Rel':>7} {'dFaith':>8} {'dRel':>7} {'Gen(ms)':>9}")
print(f"  {'-'*80}")

for pid in ["P1", "P2", "P3", "P4"]:
    r = prompt_results[pid]
    df = round(r["avg_faith"] - prompt_results["P1"]["avg_faith"], 3) if pid != "P1" else 0.0
    dr = round(r["avg_rel"]   - prompt_results["P1"]["avg_rel"],   3) if pid != "P1" else 0.0
    df_str = (f"+{df}" if df >= 0 else str(df)) if pid != "P1" else "  --"
    dr_str = (f"+{dr}" if dr >= 0 else str(dr)) if pid != "P1" else "  --"
    best_combo  = max(prompt_results[p]["avg_faith"] + prompt_results[p]["avg_rel"]
                      for p in ["P2","P3","P4"])
    this_combo  = r["avg_faith"] + r["avg_rel"]
    marker = "  <- BEST" if pid != "P1" and abs(this_combo - best_combo) < 1e-6 else ""
    print(f"  {pid:<4} {r['label']:<44} {r['avg_faith']:>7} {r['avg_rel']:>7} "
          f"{df_str:>8} {dr_str:>7} {r['avg_gen_ms']:>9}{marker}")

best_pid = max(["P2","P3","P4"],
               key=lambda p: prompt_results[p]["avg_faith"] + prompt_results[p]["avg_rel"])
print(f"\n  Best prompt config : {best_pid} -- {prompt_results[best_pid]['label'].strip()}")
print(f"  Faithfulness       : {prompt_results[best_pid]['avg_faith']}")
print(f"  Relevancy          : {prompt_results[best_pid]['avg_rel']}")
print(f"\n  Key takeaways:")
print(f"  - P2 isolates the effect of the improved GENERATION prompt")
print(f"  - P3 isolates the effect of the improved JUDGE prompts")
print(f"  - P4 combines both; use it as the new production standard if Faith+Rel > P1")
print(f"  - Improved judge (PARTIAL credit) tends to lower faithfulness scores vs binary YES/NO")
print(f"    because it catches partial claims -- this is a more honest, stricter measurement")

# Save
with open("/kaggle/working/prompt_engineering_results.json", "w") as f:
    json.dump(prompt_results, f, indent=2)
print(f"\n  Saved prompt_engineering_results.json")



  P2: Improved Gen   + Original Judge
  Q01 | Faith=1.000 | Rel=0.844 | Gen=2.405s
  Q02 | Faith=1.000 | Rel=0.730 | Gen=3.555s
  Q03 | Faith=1.000 | Rel=0.633 | Gen=4.069s
  Q04 | Faith=1.000 | Rel=0.859 | Gen=2.875s
  Q05 | Faith=1.000 | Rel=0.684 | Gen=4.93s
  Q06 | Faith=1.000 | Rel=0.750 | Gen=3.882s
  Q07 | Faith=1.000 | Rel=0.791 | Gen=2.953s
  Q08 | Faith=1.000 | Rel=0.786 | Gen=2.21s
  Q09 | Faith=1.000 | Rel=0.754 | Gen=4.589s
  Q10 | Faith=1.000 | Rel=0.711 | Gen=2.842s
  Q11 | Faith=1.000 | Rel=0.734 | Gen=3.335s
  Q12 | Faith=1.000 | Rel=0.785 | Gen=3.494s
  Q13 | Faith=1.000 | Rel=0.551 | Gen=3.54s
  Q14 | Faith=1.000 | Rel=0.795 | Gen=3.221s
  Q15 | Faith=1.000 | Rel=0.830 | Gen=3.991s

  Summary -- P2:
    Avg Faithfulness : 1.0
    Avg Relevancy    : 0.749
    Avg Gen Time     : 3459ms

  P3: Original Gen   + Improved Judge
  Q01 | Faith=1.000 | Rel=0.887 | Gen=1.011s
  Q02 | Faith=1.000 | Rel=0.887 | Gen=2.758s
  Q03 | Faith=1.000 | Rel=0.887 | Gen=2.593s
  Q04 | Fai